In [2]:
import torch
import clip
from collections import OrderedDict

# Load base CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, _ = clip.load("ViT-B/32", device=device)

# Load LoRA weights
lora_path = "./lora_modules/vitb32/cub2002011/16shots/seed1/lora_weights.pt"
lora_state = torch.load(lora_path, map_location=device)

# If saved as state_dict only
if "state_dict" in lora_state:
    lora_state = lora_state["state_dict"]

print(f"Loaded LoRA state with {len(lora_state)} tensors.")


Loaded LoRA state with 2 tensors.


/tmp/ipykernel_223238/1466048261.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  lora_state = torch.load(lora_path, map_location=device)


In [4]:
for k, v in lora_state.items():
    print(f"\nTop-level key: {k}")
    if isinstance(v, dict):
        for subk, subv in v.items():
            if torch.is_tensor(subv):
                print(f"  {k}.{subk}: {tuple(subv.shape)}, dtype={subv.dtype}")
            else:
                print(f"  {k}.{subk}: {type(subv)}")
    else:
        print(f"  {type(v)}")



Top-level key: weights
  weights.layer_0: <class 'dict'>
  weights.layer_1: <class 'dict'>
  weights.layer_2: <class 'dict'>
  weights.layer_3: <class 'dict'>
  weights.layer_4: <class 'dict'>
  weights.layer_5: <class 'dict'>
  weights.layer_6: <class 'dict'>
  weights.layer_7: <class 'dict'>
  weights.layer_8: <class 'dict'>
  weights.layer_9: <class 'dict'>
  weights.layer_10: <class 'dict'>
  weights.layer_11: <class 'dict'>
  weights.layer_12: <class 'dict'>
  weights.layer_13: <class 'dict'>
  weights.layer_14: <class 'dict'>
  weights.layer_15: <class 'dict'>
  weights.layer_16: <class 'dict'>
  weights.layer_17: <class 'dict'>
  weights.layer_18: <class 'dict'>
  weights.layer_19: <class 'dict'>
  weights.layer_20: <class 'dict'>
  weights.layer_21: <class 'dict'>
  weights.layer_22: <class 'dict'>
  weights.layer_23: <class 'dict'>

Top-level key: metadata
  metadata.r: <class 'int'>
  metadata.alpha: <class 'int'>
  metadata.encoder: <class 'str'>
  metadata.params: <class '

In [6]:
for lname, ldict in lora_state["weights"].items():
    print(f"\nLayer: {lname}")
    for subk, subv in ldict.items():
        if isinstance(subv, dict):
            for kk, vv in subv.items():
                if torch.is_tensor(vv):
                    print(f"  {lname}.{subk}.{kk}: {tuple(vv.shape)}")
        elif torch.is_tensor(subv):
            print(f"  {lname}.{subk}: {tuple(subv.shape)}")



Layer: layer_0
  layer_0.q_proj.w_lora_A: (2, 512)
  layer_0.q_proj.w_lora_B: (512, 2)
  layer_0.k_proj.w_lora_A: (2, 512)
  layer_0.k_proj.w_lora_B: (512, 2)
  layer_0.v_proj.w_lora_A: (2, 512)
  layer_0.v_proj.w_lora_B: (512, 2)

Layer: layer_1
  layer_1.q_proj.w_lora_A: (2, 512)
  layer_1.q_proj.w_lora_B: (512, 2)
  layer_1.k_proj.w_lora_A: (2, 512)
  layer_1.k_proj.w_lora_B: (512, 2)
  layer_1.v_proj.w_lora_A: (2, 512)
  layer_1.v_proj.w_lora_B: (512, 2)

Layer: layer_2
  layer_2.q_proj.w_lora_A: (2, 512)
  layer_2.q_proj.w_lora_B: (512, 2)
  layer_2.k_proj.w_lora_A: (2, 512)
  layer_2.k_proj.w_lora_B: (512, 2)
  layer_2.v_proj.w_lora_A: (2, 512)
  layer_2.v_proj.w_lora_B: (512, 2)

Layer: layer_3
  layer_3.q_proj.w_lora_A: (2, 512)
  layer_3.q_proj.w_lora_B: (512, 2)
  layer_3.k_proj.w_lora_A: (2, 512)
  layer_3.k_proj.w_lora_B: (512, 2)
  layer_3.v_proj.w_lora_A: (2, 512)
  layer_3.v_proj.w_lora_B: (512, 2)

Layer: layer_4
  layer_4.q_proj.w_lora_A: (2, 512)
  layer_4.q_proj.w_l

In [ ]:
"""

"""

import torch
import clip
from pathlib import Path

# ============================================================
# 1. Load base CLIP and LoRA checkpoint
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"

model, _ = clip.load("ViT-B/32", device=device)
print("Base CLIP loaded.")

lora_path = Path("./lora_modules/vitb32/cub2002011/16shots/seed1/lora_weights.pt")
lora_state = torch.load(lora_path, map_location=device)
layers = lora_state["weights"]
meta = lora_state["metadata"]

r, alpha = meta["r"], meta["alpha"]
scale = alpha / r
print(f"LoRA rank={r}, alpha={alpha}, scale={scale}, layers={len(layers)}")


# ============================================================
# 2. Merge helper function
# ============================================================
def apply_lora_to_block(block, lora_dict, scale):
    """
    Merges LoRA weights into q, k, v projection layers of a transformer block.
    block: CLIP transformer block
    lora_dict: dict of LoRA matrices for that block (A/B pairs)
    scale: scaling factor (alpha / r)
    """
    for proj_name in ["q_proj", "k_proj", "v_proj"]:
        try:
            A = lora_dict[f"{proj_name}.w_lora_A"].to(block.attn.in_proj_weight.device)
            B = lora_dict[f"{proj_name}.w_lora_B"].to(block.attn.in_proj_weight.device)
        except KeyError:
            print(f"Skipping {proj_name} — not found in LoRA dict")
            continue

        # Convert LoRA weight shapes (r, d) and (d, r) to (d, d)
        delta_w = scale * (B @ A)

        # Add LoRA update to base projection weight
        # Depending on your CLIP-LORA repo, projection weights may be combined (in_proj_weight)
        if hasattr(block.attn, "in_proj_weight"):  
            # Split the combined qkv weight and add to each section
            w = block.attn.in_proj_weight.data
            d = w.shape[1]
            head = d // 3
            if proj_name == "q_proj":
                w[:head, :] += delta_w
            elif proj_name == "k_proj":
                w[head:2*head, :] += delta_w
            elif proj_name == "v_proj":
                w[2*head:, :] += delta_w
        elif hasattr(block.attn, proj_name):
            block.attn.__getattr__(proj_name).weight.data += delta_w
        else:
            print(f"Could not locate {proj_name} weight in block.")


# ============================================================
# 3. Apply LoRA updates to both encoders
# ============================================================

# --- Text encoder: layers 0–11 ---
for i in range(12):
    lora_dict = layers[f"layer_{i}"]
    apply_lora_to_block(model.transformer.resblocks[i], lora_dict, scale)
print("Merged LoRA into text encoder.")

# --- Vision encoder: layers 12–23 ---
for i in range(12, 24):
    lora_dict = layers[f"layer_{i}"]
    apply_lora_to_block(model.visual.transformer.resblocks[i - 12], lora_dict, scale)
print("Merged LoRA into vision encoder.")


# ============================================================
# 4. Save merged model
# ============================================================
save_path = Path("clip_vitb32_lora_merged.pt")
torch.save(model.state_dict(), save_path)
print(f"Merged model saved to {save_path.resolve()}")


Base CLIP loaded.


/tmp/ipykernel_223238/501041443.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  lora_state = torch.load(lora_path, map_location=device)


FileNotFoundError: [Errno 2] No such file or directory: 'Documents/Concept_LoRA/lora_modules/vitb32/cub2002011/16shots/seed1/lora_weights.pt'